# P0 — GroupNorm pilot: `unet`, `unet_v2`

Round 0 of `fixing.md`'s retrain plan: validates finding #1 (`BatchNorm` at
`BATCH_SIZE=8` — well below the 50-256 range two lectures cite as typical)
on the two simplest deterministic architectures before rolling the change
out to the rest of the checkpoint tree in Round 1.

`scripts/unet.py` and `scripts/unet_v2.py` now build with
`GroupNormalization(groups=32)` in place of every `BatchNormalization`
(`_num_groups` adapts the group count down for smaller channel counts, e.g.
in unit tests — it stays 32 for this project's real channel sizes,
64-1024). This notebook trains both architectures fresh under that new
architecture and compares them against the existing BatchNorm checkpoints
in `models/deterministic/` (backed up separately in `models/pf1/` before
this branch started) — same test set, same loss, same everything except
the normalization layer.

New checkpoints save to `models/p0_groupnorm_pilot/<arch>/best_model.keras`
— a separate directory from `models/deterministic/`, so the baseline
checkpoints are never overwritten and remain available for this
comparison (and for the rest of the project) regardless of this pilot's
outcome.

Make the project root importable so `scripts.*` resolves regardless of
the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports, a fixed global seed, and a GPU sanity check — same as
`020_training.ipynb`.

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.trainer import compile_model, get_callbacks, get_model, load_model
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset — artwork-and-mockups split

Same split every training/evaluation notebook in the project uses (see
`notebooks/010_eda.ipynb` for the leakage-integrity check on it).

In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, test_pairs = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)
test_ds = build_dataset(
    test_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")
print(f"Test:  {len(test_pairs)} patches ({len(test_ds)} batches)")

## 2. Train `unet` and `unet_v2` with GroupNorm

Both use `combined_loss` (MAE + (1 - SSIM)) via `compile_model` —
unchanged from `020_training.ipynb`; the only thing that changed is what
`get_model` builds. Checkpoints go to
`models/p0_groupnorm_pilot/<arch>/best_model.keras`, keeping
`models/deterministic/` (the BatchNorm baseline) untouched. Set `EPOCHS`
lower for a quick smoke test before committing to a full run.

In [ ]:
ARCHS = ["unet", "unet_v2"]
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
MODEL_DIR = settings.MODELS_DIR / "p0_groupnorm_pilot"
LOG_DIR = settings.LOGS_DIR / "p0_groupnorm_pilot"

histories: dict = {}

for arch in ARCHS:
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {arch} (GroupNorm)")
    print(f"{'=' * 60}")

    model = get_model(arch)
    model = compile_model(
        model, arch, lr=settings.LEARNING_RATE, loss_alpha=settings.LOSS_ALPHA
    )
    model.summary(line_length=80)

    callbacks = get_callbacks(arch, log_dir=LOG_DIR, model_dir=MODEL_DIR)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    histories[arch] = history.history

    best_val_loss = min(history.history["val_loss"])
    print(f"\nBest val_loss ({arch}, GroupNorm): {best_val_loss:.4f}")

## 3. Training curves

In [ ]:
for arch, history in histories.items():
    plot_training_curves(history, title=f"Training history — {arch} (GroupNorm)")
    plt.show()

## 4. Evaluate on the test set — GroupNorm pilot vs. BatchNorm baseline

Loads the freshly trained GroupNorm checkpoints from
`models/p0_groupnorm_pilot/` alongside the existing BatchNorm checkpoints
from `models/deterministic/` (unchanged by this notebook), so both sit in
the same comparison table. A missing checkpoint on either side is skipped,
not treated as an error.

In [ ]:
BASELINE_DIR = settings.MODELS_DIR / "deterministic"
PILOT_DIR = settings.MODELS_DIR / "p0_groupnorm_pilot"

results: dict = {}

for arch in ARCHS:
    for label, model_dir in [("baseline (BatchNorm)", BASELINE_DIR), ("pilot (GroupNorm)", PILOT_DIR)]:
        key = f"{arch} — {label}"
        try:
            model = load_model(
                arch,
                model_dir=model_dir,
                lr=settings.LEARNING_RATE,
                loss_alpha=settings.LOSS_ALPHA,
            )
        except FileNotFoundError as exc:
            print(f"[skip] {exc}")
            continue

        print(f"Evaluating {key}...")
        metrics = model.evaluate(test_ds, verbose=0, return_dict=True)
        results[key] = metrics
        print(f"  {key}: { {k: f'{v:.4f}' for k, v in metrics.items()} }")

## 5. Comparison table

In [ ]:
if results:
    col_w = 28
    headers = ["model"] + list(next(iter(results.values())).keys())
    print("".join(h.ljust(col_w) for h in headers))
    print("-" * (col_w * len(headers)))
    for key, metrics in results.items():
        row = [key] + [f"{v:.4f}" for v in metrics.values()]
        print("".join(c.ljust(col_w) for c in row))

## 6. Bar chart comparison

In [ ]:
if results:
    metric_keys = list(next(iter(results.values())).keys())
    n = len(metric_keys)
    keys = list(results.keys())
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    fig.suptitle("GroupNorm pilot vs. BatchNorm baseline — test set")
    for ax, metric in zip(axes, metric_keys):
        vals = [results[k][metric] for k in keys]
        bars = ax.bar(keys, vals)
        ax.set_title(metric.upper())
        ax.set_xticklabels(keys, rotation=30, ha="right")
        for bar, v in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01,
                f"{v:.3f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )
    plt.tight_layout()
    plt.show()

## 7. Summary

Pilot checkpoints saved to `models/p0_groupnorm_pilot/<arch>/best_model.keras`.
Logs written to `logs/p0_groupnorm_pilot/<arch>/`.

**Reading the result**: if `unet`/`unet_v2` (GroupNorm) matches or beats
`unet`/`unet_v2` (BatchNorm baseline) on `mae`/`ssim`/`psnr`, Round 1 of
`fixing.md` proceeds — rolling `GroupNormalization` out to the rest of the
checkpoint tree (`resunet`, `attention_unet`, plus every `_nll` variant)
combined with the other Round 1 changes (weight decay, He init, ResUNet
identity shortcut). If it regresses, that is worth understanding before
committing the same change to 10 more checkpoints — see `fixing.md` §2 for
the full round plan.

In [ ]:
for arch in ARCHS:
    ckpt = MODEL_DIR / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25}: {status}  ({ckpt})")